# Session 12: Exploratory Data Analysis (EDA)

**Applied Machine Learning Using Python**  
**Duration**: 4 Hours (TL12)  
**Level**: 🟢 Beginner → 🟡 Intermediate

---

## 📋 What You'll Learn

| Section | Topic | Level |
|---------|-------|---------|
| 1 | Summarizing Raw Data | 🟢 |
| 2 | Visualizing Patterns and Trends | 🟢 |
| 3 | Detecting Irregularities (Outlier Removal) | 🟡 |
| 4 | Addressing Missing Values (Imputation) | 🟡 |
| 5 | Refining for Optimal Presentation | 🟡 |

In [ ]:
# Setup — Run this cell first!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ Required libraries successfully imported!')

---

## §1 — Summarizing Raw Data 🟢

We need a dataset. Let's load the famous **Titanic** dataset using Seaborn's built-in loader. This dataset is notoriously messy, making it perfect for practicing EDA.


In [ ]:
df = sns.load_dataset('titanic')

print("Shape of DataFrame:", df.shape)
display(df.head())

# 1. The .info() method instantly tells us which columns have missing values (NaNs)
print("\n--- Data Info ---")
df.info()

# 2. The .describe() method provides instant statistical summaries (mean, min, max, quartiles)
print("\n--- Statistical Summary ---")
display(df.describe())

---

## §2 — Visualizing Patterns and Trends 🟢

Visualizations are the fastest way for human brains to detect underlying patterns. Let's see if we can find out what factors most influenced whether a passenger survived.

In [ ]:
# 1. Count Plots (Categorical Data)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='sex', hue='survived', palette='Set2')
plt.title("Survival by Gender")

plt.subplot(1, 2, 2)
sns.countplot(data=df, x='pclass', hue='survived', palette='Set1')
plt.title("Survival by Passenger Class")

plt.show()

# 2. Correlation Heatmaps (Numerical Data)
plt.figure(figsize=(8, 6))
# Filter only numerical columns for correlation calculation
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='RdBu', fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

print("Pattern Found: 'pclass' (Passenger Class) is negatively correlated with survival. First class (1) survived more than Third class (3).")

---

## §3 — Detecting Irregularities (Outlier Removal) 🟡

Boxplots are the standard tool for visualizing outliers in distributions.

In [ ]:
# Step 1: Visual Detection using Boxplots
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='fare', color='#f39c12')
plt.title("Distribution of Fares (Note the extreme outliers on the right!)")
plt.show()

# Step 2: Mathematical Detection using IQR (Interquartile Range)
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1

# Anything outside of 1.5 * IQR is considered an outlier
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['fare'] < lower_bound) | (df['fare'] > upper_bound)]
print(f"Detected {len(outliers)} outliers in the 'fare' column based on IQR!")
print(f"Upper bound limit was calculated as: ${upper_bound:.2f}")

# Instead of deleting them, we can 'Cap' them (Winsorization)
df_cleaned = df.copy()
df_cleaned.loc[df_cleaned['fare'] > upper_bound, 'fare'] = upper_bound
print("\n✅ Outliers have been successfully capped to the upper bound.")

---

## §4 — Addressing Missing Values (Imputation) 🟡

Let's see exactly how much data is missing.

In [ ]:
missing_data = df_cleaned.isnull().sum()
print("Missing Values:\n", missing_data[missing_data > 0])

from sklearn.impute import SimpleImputer

# Case 1: 'deck' is missing 688 values (out of 891). This is almost 80%. 
# Solution: Delete the column entirely. It is beyond saving.
df_cleaned = df_cleaned.drop(columns=['deck'])
print("\n✅ Dropped 'deck' column.")

# Case 2: 'embark_town' is missing only 2 values.
# Solution: Simple Mode imputation (Most frequent value)
mode_imputer = SimpleImputer(strategy='most_frequent')
df_cleaned[['embark_town']] = mode_imputer.fit_transform(df_cleaned[['embark_town']])
print("✅ Imputed 'embark_town' with the most frequent value.")

# Case 3: 'age' is missing 177 values.
# Solution: Median imputation (Using median is safer than mean when there's skewness)
median_imputer = SimpleImputer(strategy='median')
df_cleaned[['age']] = median_imputer.fit_transform(df_cleaned[['age']])
print("✅ Imputed 'age' with the median age.")

# Final verification
print("\nRemaining Missing Values:", df_cleaned.isnull().sum().sum())

---

## §5 — Refining for Optimal Presentation 🟡

When presenting our EDA findings to stakeholders, visualizations must be self-explanatory, clean, and optimally formatted.

In [ ]:
# Set presentation theme
sns.set_theme(style="ticks", context="talk")

# Create beautiful subplots
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(data=df_cleaned, x='age', hue='survived', multiple="stack", 
             palette=["#e74c3c", "#2ecc71"], edgecolor=".3", linewidth=.5, bins=30, ax=ax)

# Refine Title and Labels
ax.set_title("Titanic Survival Demographics by Age", weight='bold', fontsize=18, pad=15)
ax.set_xlabel("Age of Passenger", weight='bold')
ax.set_ylabel("Number of Passengers", weight='bold')

# Refine Legend
ax.legend(title="Fate", labels=["Survived (1)", "Perished (0)"], frameon=True, loc="upper right")

# Despine removes the top and right borders for a cleaner look
sns.despine()

plt.tight_layout()
plt.show()

---

## 📝 Summary & Key Takeaways

1. **EDA** is essential for understanding your raw data before training ML models.
2. **Patterns/Trends** can be discovered rapidly using Correlation Heatmaps and Scatter/Count plots.
3. **Irregularities (Outliers)** distort distributions and pull models off course. Detect them using Z-Scores or IQR, and handle them via removal or capping.
4. **Missing Values** crash standard Machine Learning models. Handle them based on the severity: Drop entirely, Impute Mean/Median, or use advanced models (KNN) to guess them.
5. **Refining Presentations** ensures your discoveries are understood by technical and non-technical stakeholders alike.
